# v4_deepsets â€” entrenamiento en Google Colab (GitHub-based)

El cÃ³digo se clona desde GitHub en cada sesiÃ³n; el dataset se baja de Drive.

## Setup previo (una sola vez)
1. SubÃ­ el HDF5 a Google Drive en `MyDrive/Ipre/datasets/v2_xyz100_step50_n5000.h5`.
2. Colab â†’ Ã­cono ðŸ”‘ *Secrets* (panel izquierdo) â†’ agregÃ¡ `COMET_API_KEY` con tu API key de comet.com. ActivÃ¡ *Notebook access*.
3. *Runtime â†’ Change runtime type â†’ GPU* (T4 free / A100 Pro).

## Cada sesiÃ³n
CorrÃ© las celdas de arriba a abajo. La sesiÃ³n arranca limpia â†’ siempre clonÃ¡s repo + copiÃ¡s dataset.

## 1. GPU

In [ ]:
!nvidia-smi

## 2. Clonar el repo desde GitHub

In [ ]:
REPO_URL = 'https://github.com/Daspony/PMDKernel.git'
BRANCH   = 'main'

import os
if os.path.isdir('PMDKernel'):
    %cd PMDKernel
    !git fetch origin && git checkout {BRANCH} && git pull origin {BRANCH}
else:
    !git clone --branch {BRANCH} {REPO_URL}
    %cd PMDKernel
print('cwd:', os.getcwd())
!git log -1 --oneline

## 3. Instalar dependencias

Colab ya trae `torch` con CUDA. Solo falta el resto.

In [ ]:
!pip install -q pytorch-lightning comet-ml h5py torchmetrics

In [ ]:
import torch, pytorch_lightning as pl, comet_ml, h5py, sklearn, torchmetrics
print(f'torch        = {torch.__version__}  cuda={torch.cuda.is_available()}')
print(f'lightning    = {pl.__version__}')
print(f'comet_ml     = {comet_ml.__version__}')
print(f'h5py         = {h5py.__version__}')
print(f'torchmetrics = {torchmetrics.__version__}')
if torch.cuda.is_available():
    print(f'device       = {torch.cuda.get_device_name(0)}')

## 4. Bajar el dataset desde Drive

Monto Drive solo para copiar el H5 al directorio local del runtime. DespuÃ©s podÃ©s desmontar si querÃ©s.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
DATASET = 'v2_xyz100_step50_n5000.h5'
DRIVE_SRC = f'/content/drive/MyDrive/Ipre/datasets/{DATASET}'

!mkdir -p data/datasets
!cp -v {DRIVE_SRC} data/datasets/
!ls -lh data/datasets/

## 5. Credenciales Comet

Lee `COMET_API_KEY` del panel de Secrets de Colab. Si lo dejÃ¡s vacÃ­o, podÃ©s desactivar el logger pasando `os.environ['COMET_OFFLINE'] = '1'`.

In [ ]:
from google.colab import userdata
import os

os.environ['COMET_API_KEY']   = userdata.get('COMET_API_KEY')
os.environ['COMET_WORKSPACE'] = 'sebasti-n-vallejos'
print('comet workspace:', os.environ['COMET_WORKSPACE'])
print('api key len:', len(os.environ['COMET_API_KEY']))

## 6. Entrenar

EditÃ¡ `RUN_TAG`, `EPOCHS` si querÃ©s.

In [ ]:
H5            = f'data/datasets/{DATASET}'
RUN_TAG       = 'v4_deepsets_v2_n5000_colab'
EPOCHS        = 100
PATIENCE      = 20
BATCH_SIZE    = 1024
COMET_PROJECT = 'pmdkernel'
CKPT_DIR      = f'/content/drive/MyDrive/Ipre/runs/{RUN_TAG}'  # escribe directo a Drive

!python python/train_v4.py \
    --h5 {H5} \
    --run-tag {RUN_TAG} \
    --ckpt-dir {CKPT_DIR} \
    --epochs {EPOCHS} \
    --patience {PATIENCE} \
    --batch-size {BATCH_SIZE} \
    --comet-project {COMET_PROJECT} \
    --no-progress \
    --num-workers 2 \
    --pin-memory

### Smoke test (3 epochs)

In [ ]:
!python python/train_v4.py \
    --h5 data/datasets/{DATASET} \
    --run-tag v4_deepsets_colab_smoke \
    --quick \
    --no-progress \
    --num-workers 2 \
    --pin-memory

## 7. Verificar que el ckpt persistió en Drive

Como pasamos `--ckpt-dir` apuntando directo a Drive, Lightning escribió `<run_tag>.ckpt`, `last.ckpt` y `<run_tag>_aux.pt` ahí durante el train. No hace falta `cp` al final.

In [ ]:
!ls -lh {CKPT_DIR}/

## 8. Inspeccionar resultados

In [ ]:
import torch
from pathlib import Path

aux_path = Path(CKPT_DIR) / f'{RUN_TAG}_aux.pt'
aux = torch.load(aux_path, weights_only=False)

print('checkpoint:', aux['ckpt_path'])
print('comet url :', aux['comet_url'])
print()
for split, m in aux['metrics'].items():
    print(f"{split:5s}  rmse={m['rmse_mt']:.4f} mT  r2={m['r2']:.4f}")

## 9. Inferencia sobre una muestra del test split

Equivalente a las celdas finales de `analisis.ipynb`. Como aun no hay un `v2_test_og.h5` independiente, usamos un sample del **test split** del mismo H5 - son 750 muestras (15%) que el modelo nunca vio durante training (split determinista con `seed=42`).

In [ ]:
import sys, os, torch, numpy as np, h5py
from pathlib import Path

PROJECT = '/content/PMDKernel'
os.chdir(PROJECT)

for cand in ('python', 'Python'):
    p = os.path.join(PROJECT, cand)
    if os.path.isdir(p):
        if p not in sys.path:
            sys.path.insert(0, p)
        break
else:
    raise RuntimeError('no se encontro python/ ni Python/')

from Models.v4_deepsets.model import LitDeepSet

ckpt_path = Path(CKPT_DIR) / f'{RUN_TAG}.ckpt'
if not ckpt_path.exists():
    cands = sorted(Path(CKPT_DIR).glob(f'{RUN_TAG}*.ckpt'))
    cands = [c for c in cands if 'last' not in c.name]
    ckpt_path = cands[-1]
aux_path = Path(CKPT_DIR) / f'{RUN_TAG}_aux.pt'

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model  = LitDeepSet.load_from_checkpoint(str(ckpt_path), map_location=device, weights_only=False).eval().to(device)
aux    = torch.load(aux_path, map_location='cpu', weights_only=False)

print(f"ckpt         : {ckpt_path.name}")
print(f"device       : {device}")
print(f"train h5     : {aux['h5_path']}")
print(f"sensor_xyz   : {aux['sensor_xyz'].shape}")
print(f"b_mean,b_std : {aux['b_mean']:.4f}, {aux['b_std']:.4f} mT  (del training)")
m_te = aux['metrics']['test']
print(f"training test: RMSE={m_te['rmse_mt']:.4e} mT  R2={m_te['r2']:+.4f}")

In [ ]:
# Reconstruir el split test del training para tomar un sample que el modelo NO vio
from Models.v4_deepsets import data as data_mod

H5_FULL = 'data/datasets/' + DATASET
ds      = data_mod.load_dataset_metadata(H5_FULL)
splits  = data_mod.split_indices(ds['N'], val_frac=0.15, test_frac=0.15, seed=42)
test_idx = splits['test']
print(f"n_test = {len(test_idx)} muestras  (sample ids: {test_idx[:5]}...)")

TEST_POS = 0  # 0..n_test-1 - cambia para mirar otra muestra
sample_id = int(test_idx[TEST_POS])

with h5py.File(H5_FULL, 'r') as f:
    sensors_by = f['geometria/sens/B'][sample_id][..., 1].astype(np.float32)  # (I,) mT
    R_grid     = f['geometria/grid/R'][:].astype(np.float32)                   # (J, 3) mm
    by_true    = f['geometria/grid/B'][sample_id][..., 1].astype(np.float32)   # (J,) mT
    gx = f['geometria/grid/meta/x'][:].astype(np.float32)
    gy = f['geometria/grid/meta/y'][:].astype(np.float32)
    gz = f['geometria/grid/meta/z'][:].astype(np.float32)

Nx, Ny, Nz = len(gx), len(gy), len(gz)
J = R_grid.shape[0]
I = sensors_by.shape[0]
print(f"sample id     : {sample_id}  (pos {TEST_POS} en test split)")
print(f"grid          : {Nx}x{Ny}x{Nz} = {J} puntos")
print(f"sensores By   : min={sensors_by.min():.3f}  max={sensors_by.max():.3f}  std={sensors_by.std():.3f}  mT")
print(f"By true       : min={by_true.min():.3f}  max={by_true.max():.3f}  mT")

In [ ]:
# Forward sobre los J puntos. Con J=125 no hace falta chunked.
sensors_rep = np.broadcast_to(sensors_by, (J, I)).copy()   # (J, I)
sensors_t   = torch.from_numpy(sensors_rep).to(device)
query_t     = torch.from_numpy(R_grid).to(device)

with torch.no_grad():
    by_pred = model.predict_mT(sensors_t, query_t).squeeze(-1).cpu().numpy()

err  = by_pred - by_true
rmse = float(np.sqrt(np.mean(err**2)))
mae  = float(np.abs(err).mean())
maxe = float(np.abs(err).max())
r2   = float(1 - (err**2).sum() / ((by_true - by_true.mean())**2).sum())

print(f"RMSE      : {rmse:.4e} mT")
print(f"MAE       : {mae:.4e} mT")
print(f"max |err| : {maxe:.4e} mT   ({100*maxe/np.abs(by_true).max():.2f}% del |By| max)")
print(f"R2        : {r2:+.4f}")

# Inspeccion punto por punto (J=125 cabe entero, mostramos primeros 10)
print()
print('idx   By_true     By_pred     err')
for j in range(min(J, 10)):
    print(f"{j:3d}   {by_true[j]:+.3f}   {by_pred[j]:+.3f}   {err[j]:+.4f}  mT")